In [2]:
import pandas as pd

# =====================================================
# 1. Wczytanie danych
# =====================================================
df = pd.read_csv("climate_deforestation_supervised_modeling_base.csv")

# =====================================================
# 2. Agregacja miesięcy -> lata
# =====================================================
annual = (
    df.groupby(["State", "Year"])
      .agg(
          Annual_Deforestation=("Deforestation_ha", "sum"),
          Mean_Temperature=("Air_Temperature", "mean"),
          Mean_Precipitation=("Total_Precipitation", "mean"),
          Mean_Humidity=("Relative_Humidity", "mean"),
          Mean_Radiation=("Global_Radiation", "mean")
      )
      .reset_index()
)

# =====================================================
# 3. Roczna anomalia temperatury
# =====================================================
state_mean_temp = (
    annual.groupby("State")["Mean_Temperature"]
          .mean()
)

annual["Temperature_Anomaly"] = (
    annual["Mean_Temperature"]
    - annual["State"].map(state_mean_temp)
)

# =====================================================
# 4. Sortowanie
# =====================================================
annual = annual.sort_values(
    ["State", "Year"]
).reset_index(drop=True)

group = annual.groupby("State")

# =====================================================
# 5. Opóźnienia wylesienia
# =====================================================
annual["Defor_t"] = annual["Annual_Deforestation"]

annual["Defor_t-1"] = group["Annual_Deforestation"].shift(1)
annual["Defor_t-2"] = group["Annual_Deforestation"].shift(2)
annual["Defor_t-3"] = group["Annual_Deforestation"].shift(3)

# =====================================================
# 6. Skumulowane wylesienie
# (narastająco od początku)
# =====================================================
annual["Cumulative_Deforestation"] = (
    group["Annual_Deforestation"]
         .cumsum()
)

# =====================================================
# 7. 4-letnia suma krocząca
# =====================================================
annual["Rolling4_Deforestation"] = (
    group["Annual_Deforestation"]
         .rolling(window=4,
                  min_periods=1)
         .sum()
         .reset_index(level=0, drop=True)
)

# =====================================================
# 8. Ważone wylesienie
# =====================================================
annual["Weighted_Deforestation"] = (

      0.40 * annual["Defor_t"]

    + 0.30 * annual["Defor_t-1"]

    + 0.20 * annual["Defor_t-2"]

    + 0.10 * annual["Defor_t-3"]

)

# =====================================================
# 9. Target
# anomalia temperatury rok później
# =====================================================
annual["Target"] = (
    group["Temperature_Anomaly"]
         .shift(-1)
)

# =====================================================
# 10. Usunięcie braków
# =====================================================
annual = annual.dropna().reset_index(drop=True)

# =====================================================
# 11. Podział czasowy
# =====================================================
train = annual[annual["Year"] <= 2018].copy()
test = annual[annual["Year"] > 2018].copy()

# =====================================================
# 12. Features
# =====================================================
features = [

    "Year",

    "Annual_Deforestation",

    "Defor_t-1",

    "Defor_t-2",

    "Defor_t-3",

    "Cumulative_Deforestation",

    "Rolling4_Deforestation",

    "Weighted_Deforestation",

    "Mean_Precipitation",

    "Mean_Humidity",

    "Mean_Radiation"

]

X_train = train[features]
y_train = train["Target"]

X_test = test[features]
y_test = test["Target"]

# =====================================================
# 13. Zapis
# =====================================================
annual.to_csv(
    "prepared_annual_dataset.csv",
    index=False
)

print("="*50)
print("Dataset przygotowany")
print("="*50)
print("Liczba rekordów:", len(annual))
print("Train:", len(train))
print("Test :", len(test))
print("\nFeatures:")
for f in features:
    print("-", f)

print("\nTarget: Temperature_Anomaly następnego roku")

Dataset przygotowany
Liczba rekordów: 392
Train: 293
Test : 99

Features:
- Year
- Annual_Deforestation
- Defor_t-1
- Defor_t-2
- Defor_t-3
- Cumulative_Deforestation
- Rolling4_Deforestation
- Weighted_Deforestation
- Mean_Precipitation
- Mean_Humidity
- Mean_Radiation

Target: Temperature_Anomaly następnego roku


In [3]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(max_depth=12, min_samples_leaf=5, n_estimators=500,
                      n_jobs=-1, random_state=42)

In [4]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Predykcja
y_pred = rf.predict(X_test)

# Metryki
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R²   = {r2:.4f}")
print(f"MAE  = {mae:.4f}")

R²   = -0.0590
MAE  = 0.6204
